# Rock-Paper-Scissors
Training a CNN on rock-paper-scissors hand images to build a simple game application

In [ ]:
# import modules -----------------------------------------------------------------------------------
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
#from keras import callbacks
#from keras.preprocessing import image
#from keras.preprocessing.image import ImageDataGenerator
#from keras.models import Sequential
#from keras.layers import InputLayer, Dropout, Flatten, Dense, Conv2D, MaxPooling2D
#from tensorflow.keras.models import load_model
from tensorflow.keras import callbacks
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import InputLayer, Dropout, Flatten, Dense, Conv2D, MaxPooling2D
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# define paths -------------------------------------------------------------------------------------
train_folder = '../dataset_wlx/train'
test_folder = '../dataset_wlx/test'
validation_folder = '../dataset_wlx/validation'

# Helper methods -----------------------------------------------------------------------------------
'''def load_image(img_path: str) -> np.ndarray:
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return image_rgb'''
    
def load_image(img_path: str) -> np.ndarray:
    """Load an image and return grayscale 64x64 (H,W,1)."""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)     # (H,W)
    img = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)
    img = img[..., np.newaxis]                           # (64,64,1)
    return img

def display_image(image: np.ndarray) -> None:
    plt.imshow(image.squeeze(), cmap='gray')
    plt.axis('off')
    plt.show()

def save_rps_model(model: Sequential, name: str) -> None:
    '''Save a model to models/<name>.h5.'''
    filename = f'models/{name}.h5'
    model.save(filename)

def load_rps_model(model_path: str = None):
    '''Load a model from a file path. If no path is provided, the latest model is loaded.'''

    if not os.path.isfile(str(model_path)):
        # load latest version
        model_path = os.path.join('models', sorted([m for m in os.listdir('models') if m.lower().endswith('.h5')])[-1])

    model = load_model(model_path)
    #input_shape = model.layers[0].input_shape

    #print(f'Loaded model: {model_path} with input shape: {input_shape}')
    input_shape = model.input_shape
    print(f'Loaded model: {model_path} with input shape: {input_shape}')


    return model

def get_random_picture(set: str = None) -> str:
    '''Return a random picture path from the train, test or validation set.'''

    if set == 'train':
        folder = train_folder
    elif set == 'test':
        folder = test_folder
    elif set == 'validation':
        folder = validation_folder
    else:
        folder = random.choice([train_folder, test_folder, validation_folder])
    
    subfolder = random.choice(os.listdir(folder))
    subfolder_path = os.path.join(folder, subfolder)
    test_image = random.choice(os.listdir(subfolder_path))
    image_path = os.path.normpath(os.path.join(subfolder_path, test_image))

    return image_path

## Check dataset content

In [ ]:
# open test picture
image_path = get_random_picture()
print('Image:', image_path)
img = load_image(image_path)
display_image(img)

## Preparing data for the model

In [ ]:
image_shape = (64, 64, 1)
batch_size = 32

image_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='nearest'
)

train_image_gen = image_gen.flow_from_directory(
    train_folder,
    target_size=image_shape[:2],
    color_mode='grayscale',          
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

test_image_gen = image_gen.flow_from_directory(
    test_folder,
    target_size=image_shape[:2],
    color_mode='grayscale',          
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

x, y = next(train_image_gen)
print("x:", x.shape, "y:", y.shape)

plt.imshow(x[0].squeeze(), cmap='gray')
plt.title(f"label={np.argmax(y[0])}")
plt.axis('off')
plt.show()

validation_generator = image_gen.flow_from_directory(
    validation_folder,
    target_size=image_shape[:2],
    color_mode='grayscale',          # ✅
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# test generator on random image
random_img = image_gen.random_transform(img)
display_image(random_img)

## Create Model

In [ ]:
# create sequential model
#3x3 filter1=4, filter2=8, filter3=8
model = Sequential()

# input: 64x64x1
model.add(InputLayer(input_shape=(64, 64, 1)))

# Conv1: 3x3, Cin=1, Cout=4   => 62x62x4
model.add(Conv2D(filters=4, kernel_size=(3, 3), strides=(1, 1),
                 padding='valid', activation='relu'))
# Pool1: 2x2 => 31x31x4
model.add(MaxPooling2D(pool_size=(2, 2)))

# Conv2: 3x3, Cin=4, Cout=8   => 29x29x8pooling
model.add(Conv2D(filters=8, kernel_size=(3, 3), strides=(1, 1),
                 padding='valid', activation='relu'))
# Pool2: 2x2 => 14x14x8
model.add(MaxPooling2D(pool_size=(2, 2)))
# Conv3: 3x3, Cin=8, Cout=8   => 12x12x8pooling
model.add(Conv2D(filters=8, kernel_size=(3, 3), strides=(1, 1),
                 padding='valid', activation='relu'))
# Pool3: 2x2 => 6x6x8
model.add(MaxPooling2D(pool_size=(2, 2)))
# Flatten => 576
model.add(Flatten())

# FC: 576 --> 3
model.add(Dense(units=3, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

## Training Model

In [ ]:
# indices
class_indices = train_image_gen.class_indices
class_labels = {v:k for k, v in class_indices.items()}
print(class_indices)
print(class_labels)

In [ ]:
# Initializing early stopping callback to monitor the validation loss and prevent overfitting
earlystopping = callbacks.EarlyStopping(monitor="accuracy",
                                        mode="max",
                                        patience=200,
                                        restore_best_weights=True)

In [ ]:
# Train the model
results = model.fit(
    train_image_gen,
    epochs=200,
    steps_per_epoch=train_image_gen.samples//train_image_gen.batch_size,
    validation_data=test_image_gen,
    validation_steps=test_image_gen.samples//test_image_gen.batch_size,
    verbose=2,
    callbacks=[earlystopping]
    )

## Visualize Accuracy

In [ ]:
plt.plot(results.history['accuracy'], label='Train Accuracy')
plt.plot(results.history['val_accuracy'], label='Validation Accuracy')

plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()


## Save model

In [ ]:
save_rps_model(model, 'v2')

## Export INT8 TFLite (PTQ)
Use post-training quantization (PTQ) to export a full-int8 `.tflite` model directly in this notebook.


In [ ]:
from pathlib import Path
import subprocess
import sys
import textwrap

# 1) save current floating model (optional)
# save_rps_model(model, '64x64_floating_v1')

# 2) find latest saved floating model
candidates = sorted(Path('models').glob('v2_acc=92.8.h5'))
if not candidates:
    raise FileNotFoundError('No saved floating model found under models/')
model_h5 = candidates[-1]

# Optional custom output model filename (without extension)
# Example: custom_output_name = '64x64_floating_v1_int8_balanced'
custom_output_name = "v2_int8"

if custom_output_name:
    out_path = Path('models') / f"{custom_output_name}.int8.tflite"
else:
    out_path = Path('models') / f"{model_h5.stem}.int8.tflite"
print(f'[INFO] Source model: {model_h5}')

# PTQ representative dataset config
per_class_samples = 2000  # 每类采样数量，可改（例如 100/200/300）
print(f'[INFO] representative per-class samples: {per_class_samples}')

# 3) run PTQ conversion in subprocess to avoid kernel crash
script = textwrap.dedent("""
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf

model_path = Path(sys.argv[1])
rep_dir = Path(sys.argv[2])
out_path = Path(sys.argv[3])
per_class_samples = int(sys.argv[4])

# Try loading full model first; fallback to manual architecture for cross-version compatibility
try:
    float_model = tf.keras.models.load_model(str(model_path), compile=False)
except Exception as e:
    print('[WARN] load_model failed, fallback to manual architecture + load_weights:', e)
    float_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(64, 64, 1)),
        tf.keras.layers.Conv2D(filters=4, kernel_size=(3, 3), strides=(1, 1), padding='valid', activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(filters=8, kernel_size=(3, 3), strides=(1, 1), padding='valid', activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(filters=8, kernel_size=(3, 3), strides=(1, 1), padding='valid', activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units=3, activation='softmax'),
    ])
    float_model.build((None, 64, 64, 1))
    float_model.load_weights(str(model_path))

def representative_dataset_gen():
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    class_dirs = sorted([d for d in rep_dir.iterdir() if d.is_dir()])
    if not class_dirs:
        raise FileNotFoundError(f'No class subdirs found in {rep_dir}')

    picked = []
    class_counts = {}
    for d in class_dirs:
        files = [p for p in sorted(d.rglob('*')) if p.suffix.lower() in exts]
        if not files:
            class_counts[d.name] = 0
            continue
        take = min(per_class_samples, len(files))
        picked.extend(files[:take])
        class_counts[d.name] = take

    total = len(picked)
    if total == 0:
        raise FileNotFoundError(f'No representative images found in {rep_dir}')

    print('[INFO] representative class counts:', class_counts)
    print('[INFO] representative total:', total)

    for p in picked:
        img = tf.keras.utils.load_img(p, color_mode='grayscale', target_size=(64, 64))
        arr = tf.keras.utils.img_to_array(img).astype(np.float32) / 255.0
        arr = np.expand_dims(arr, axis=0)
        yield [arr]

converter = tf.lite.TFLiteConverter.from_keras_model(float_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
out_path.write_bytes(tflite_model)

interpreter = tf.lite.Interpreter(model_content=tflite_model)
in_detail = interpreter.get_input_details()[0]
out_detail = interpreter.get_output_details()[0]

print(f'[OK] Saved int8 model: {out_path}')
print('[INFO] Input dtype/quant:', in_detail['dtype'].__name__, in_detail['quantization'])
print('[INFO] Output dtype/quant:', out_detail['dtype'].__name__, out_detail['quantization'])
""")

preferred_py = Path('.venv_tflite/bin/python')
runner = str(preferred_py) if preferred_py.exists() else sys.executable
print(f'[INFO] Python runner: {runner}')
if not preferred_py.exists():
    print('[WARN] .venv_tflite not found, fallback to current kernel python.')

cmd = [runner, '-c', script, str(model_h5), '../dataset_wlx/train', str(out_path), str(per_class_samples)]
print('[INFO] Running conversion in subprocess...')
res = subprocess.run(cmd, capture_output=True, text=True)

print('----- STDOUT -----')
print(res.stdout if res.stdout else '<empty>')
print('----- STDERR -----')
print(res.stderr if res.stderr else '<empty>')

if res.returncode != 0:
    raise RuntimeError(f'PTQ conversion failed (return code={res.returncode}). See logs above.')
print('[OK] PTQ conversion finished.')


## Confusion Matrix and Classification Report

In [ ]:
model = load_rps_model('models/v2_acc=92.8.h5')
validation_steps = len(validation_generator)
validation_generator.reset() # Reset the generator to be sure of avoiding shuffling

#predictions = model.predict_generator(validation_generator, steps=validation_steps, verbose=1)
import math
import numpy as np

validation_generator.reset()  

validation_steps = math.ceil(validation_generator.samples / validation_generator.batch_size)

predictions = model.predict(validation_generator, steps=validation_steps, verbose=1)
y_pred_classes = np.argmax(predictions, axis=1)

#y_pred_classes = np.argmax(predictions, axis=1)

# Print classification report
print("Classification Report:")
print(classification_report(validation_generator.classes, y_pred_classes))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(validation_generator.classes, y_pred_classes))

## Classify Validation Pictures
Run all validation pictures through the model and classify them. Display the failed ones.

In [ ]:
input_model_size = tuple(model.input_shape[1:3])
print("Model input size:", input_model_size)

success = 0
failure = 0
total = 0
failed_images = []

class_indices = validation_generator.class_indices if 'validation_generator' in globals() else {'paper':0,'rock':1,'scissors':2}
class_labels = {v: k for k, v in class_indices.items()}
print("class_indices:", class_indices)

test_images = None
idx = 0

for subfolder in ['rock', 'paper', 'scissors']:
    current_folder = os.path.join(validation_folder, subfolder)

    for img_name in os.listdir(current_folder):
        if test_images is not None and idx > test_images - 1:
            break
        idx += 1

        img_file = os.path.normpath(os.path.join(current_folder, img_name))

        img_pil = image.load_img(img_file, target_size=input_model_size, color_mode='grayscale')
        img_array = image.img_to_array(img_pil)          # (64,64,1)

        img_normalized = img_array / 255.0
        img_expanded = np.expand_dims(img_normalized, axis=0)  # (1,64,64,1)

        prediction_prob = model.predict(img_expanded, verbose=0)  # (1,3)

        
        predicted_class_index = int(np.argmax(prediction_prob[0], axis=-1))
        predicted_class_label = class_labels[predicted_class_index]

        if subfolder == predicted_class_label:
            success += 1
        else:
            failure += 1
           
            failed_images.append((img_normalized, predicted_class_label, img_file))

        total += 1
        to_print = f'Images Classified: {total} | Success: {success} | Failure: {failure} | Accuracy: {success/total*100:.2f}%'
        print(to_print, end='\r', flush=True)

print()  # 换行

# print failed images with prediction and path
#if failed_images:
 #   print(f"\n{50 * '-'}\nFAILED IMAGES:")
  #  for img_norm, label, path in failed_images:
   #     print(path)
    #   plt.title(label, loc='left')
     #   plt.axis('off')
      #  plt.show()

## Compare the int8 model result with the floating32 model

In [7]:
from pathlib import Path
import numpy as np
import tensorflow as tf

# ====== config ======
val_root = Path("../dataset_wlx/validation")
h5_path = Path("models/v2.h5")
tflite_path = Path("models/v2.int8.tflite")  # 改成你的文件
thumb_size = (64, 64)

# ====== class setup ======
class_names = sorted([d.name for d in val_root.iterdir() if d.is_dir()])
class_to_idx = {c: i for i, c in enumerate(class_names)}
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

samples = []
for cname in class_names:
    for p in sorted((val_root / cname).iterdir()):
        if p.suffix.lower() in exts:
            samples.append((p, class_to_idx[cname]))

print("[INFO] classes:", class_names)
print("[INFO] total samples:", len(samples))
print("[INFO] h5:", h5_path)
print("[INFO] tflite:", tflite_path)

# ====== float model ======
float_model = tf.keras.models.load_model(str(h5_path), compile=False)

# ====== int8 tflite ======
interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
in_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
in_scale, in_zero = in_d["quantization"]
out_scale, out_zero = out_d["quantization"]

print("[INFO] tflite input quant:", in_d["quantization"])
print("[INFO] tflite output quant:", out_d["quantization"])

# ====== eval both ======
y_true = []
y_pred_float = []
y_pred_int8 = []

for p, true_idx in samples:
    # same preprocessing for both
    img = tf.keras.utils.load_img(p, color_mode="grayscale", target_size=thumb_size)
    x = tf.keras.utils.img_to_array(img).astype(np.float32) / 255.0   # (64,64,1)
    x = x[np.newaxis, ...]  # (1,64,64,1)

    # float
    y_f = float_model.predict(x, verbose=0)[0]
    pf = int(np.argmax(y_f))

    # int8
    x_q = np.round(x / in_scale + in_zero).astype(np.int8)
    x_q = np.clip(x_q, -128, 127)
    interp.set_tensor(in_d["index"], x_q)
    interp.invoke()
    y_q = interp.get_tensor(out_d["index"])[0]
    y_i = (y_q.astype(np.float32) - out_zero) * out_scale
    pi = int(np.argmax(y_i))

    y_true.append(true_idx)
    y_pred_float.append(pf)
    y_pred_int8.append(pi)

y_true = np.array(y_true)
y_pred_float = np.array(y_pred_float)
y_pred_int8 = np.array(y_pred_int8)

acc_f = (y_true == y_pred_float).mean()
acc_i = (y_true == y_pred_int8).mean()

print(f"\n[RESULT] float acc = {acc_f:.4f}")
print(f"[RESULT] int8  acc = {acc_i:.4f}")
print(f"[RESULT] delta(int8-float) = {acc_i - acc_f:+.4f}")

# agreement float vs int8
agree = (y_pred_float == y_pred_int8).mean()
print(f"[RESULT] float/int8 prediction agreement = {agree:.4f}")

# per-class
print("\n[PER-CLASS]")
for cname, idx in class_to_idx.items():
    m = (y_true == idx)
    af = (y_pred_float[m] == idx).mean() if m.any() else 0.0
    ai = (y_pred_int8[m] == idx).mean() if m.any() else 0.0
    print(f"{cname:<10} n={m.sum():<4} float={af:.4f} int8={ai:.4f} delta={ai-af:+.4f}")


[INFO] classes: ['paper', 'rock', 'scissors']
[INFO] total samples: 850
[INFO] h5: models/v2.h5
[INFO] tflite: models/v2.int8.tflite
[INFO] tflite input quant: (0.003921568859368563, -128)
[INFO] tflite output quant: (0.00390625, -128)

[RESULT] float acc = 0.9282
[RESULT] int8  acc = 0.9329
[RESULT] delta(int8-float) = +0.0047
[RESULT] float/int8 prediction agreement = 0.9906

[PER-CLASS]
paper      n=295  float=0.9322 int8=0.9322 delta=+0.0000
rock       n=271  float=0.9557 int8=0.9557 delta=+0.0000
scissors   n=284  float=0.8979 int8=0.9120 delta=+0.0141


## FINAL TESTING
Define a ROI from webcam frames and predict the class.

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf

# ====== config ======
val_root = Path("../dataset_wlx/validation")
h5_path = Path("models/64x64_floating_v1.h5")
tflite_path = Path("models/64x64_floating_v1_int8_balanced_2000.int8.tflite")  # 改成你的文件
thumb_size = (64, 64)

# ====== class setup ======
class_names = sorted([d.name for d in val_root.iterdir() if d.is_dir()])
class_to_idx = {c: i for i, c in enumerate(class_names)}
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

samples = []
for cname in class_names:
    for p in sorted((val_root / cname).iterdir()):
        if p.suffix.lower() in exts:
            samples.append((p, class_to_idx[cname]))

print("[INFO] classes:", class_names)
print("[INFO] total samples:", len(samples))
print("[INFO] h5:", h5_path)
print("[INFO] tflite:", tflite_path)

# ====== float model ======
float_model = tf.keras.models.load_model(str(h5_path), compile=False)

# ====== int8 tflite ======
interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
in_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
in_scale, in_zero = in_d["quantization"]
out_scale, out_zero = out_d["quantization"]

print("[INFO] tflite input quant:", in_d["quantization"])
print("[INFO] tflite output quant:", out_d["quantization"])

# ====== eval both ======
y_true = []
y_pred_float = []
y_pred_int8 = []

for p, true_idx in samples:
    # same preprocessing for both
    img = tf.keras.utils.load_img(p, color_mode="grayscale", target_size=thumb_size)
    x = tf.keras.utils.img_to_array(img).astype(np.float32) / 255.0   # (64,64,1)
    x = x[np.newaxis, ...]  # (1,64,64,1)

    # float
    y_f = float_model.predict(x, verbose=0)[0]
    pf = int(np.argmax(y_f))

    # int8
    x_q = np.round(x / in_scale + in_zero).astype(np.int8)
    x_q = np.clip(x_q, -128, 127)
    interp.set_tensor(in_d["index"], x_q)
    interp.invoke()
    y_q = interp.get_tensor(out_d["index"])[0]
    y_i = (y_q.astype(np.float32) - out_zero) * out_scale
    pi = int(np.argmax(y_i))

    y_true.append(true_idx)
    y_pred_float.append(pf)
    y_pred_int8.append(pi)

y_true = np.array(y_true)
y_pred_float = np.array(y_pred_float)
y_pred_int8 = np.array(y_pred_int8)

acc_f = (y_true == y_pred_float).mean()
acc_i = (y_true == y_pred_int8).mean()

print(f"\n[RESULT] float acc = {acc_f:.4f}")
print(f"[RESULT] int8  acc = {acc_i:.4f}")
print(f"[RESULT] delta(int8-float) = {acc_i - acc_f:+.4f}")

# agreement float vs int8
agree = (y_pred_float == y_pred_int8).mean()
print(f"[RESULT] float/int8 prediction agreement = {agree:.4f}")

# per-class
print("\n[PER-CLASS]")
for cname, idx in class_to_idx.items():
    m = (y_true == idx)
    af = (y_pred_float[m] == idx).mean() if m.any() else 0.0
    ai = (y_pred_int8[m] == idx).mean() if m.any() else 0.0
    print(f"{cname:<10} n={m.sum():<4} float={af:.4f} int8={ai:.4f} delta={ai-af:+.4f}")
